# Getting Started: Understanding StudyState and the Function Registry with a Minimal Analysis

In empirical research, the work that actually takes time is rarely model fitting itself—it's aligning scattered pieces: which column in your data is the treatment, which is the outcome, which design strategy identifies causality, what assumptions must be checked before employing that design, and which step or dataset produced each final number. `socialverse` is an analysis library built for social science. Its approach makes this "alignment work" explicit: a `StudyState` object captures every dimension of your analysis (data, design, assumptions, models, diagnostics, …), and a function registry records what each method requires as inputs and what it produces, so you verify dependencies before acting—rather than guessing function names and call sequences from memory.

This tutorial avoids jargon, instead walking you through a minimal but real analysis—using difference-in-differences (DID) to estimate a policy effect—while introducing two core concepts: `StudyState` is the social-science counterpart to AnnData from bioinformatics (but it unifies **vocabulary** rather than data), and the registry is like a methods handbook that sequences your workflow and catches errors. By the end, you'll see why organizing an analysis as "state + registry" makes it harder to get wrong and easier for others to reproduce.

We use a built-in synthetic panel: 40 firms over 8 years, half exposed to a policy partway through, with a true effect set to −0.8. It's small enough to trace every step. For methodological background, consult standard DID references (such as the `fixest` or `did` R packages); the focus here is how the tools are organized.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import socialverse as sv
from socialverse import datasets as ds

## Learning the Registry: Where Methods Are Listed

When you import `socialverse`, functions across analysis modules automatically register with a process-level registry, `sv.registry`. Think of it as a catalog of the library: how many methods exist and how they are categorized. The categories follow the real workflow of social science—causal inference, complex survey analysis, psychometrics, text and qualitative analysis, networks, demography, governance and compliance, and more. Before starting an analysis, it's good practice to scan the catalog first.

In [2]:
print(len(sv.registry), "个已登记的方法")
print("类别:", sv.registry.categories())

54 个已登记的方法
类别: ['causal', 'demography', 'econ', 'figure', 'governance', 'lens', 'literature', 'longitudinal', 'net', 'prep', 'psychometrics', 'qual', 'quasi', 'setmethods', 'spatial', 'stylometry', 'survey', 'text']


## Finding Methods by Name: `find`

Suppose you have a research question—"I want to fit difference-in-differences"—but can't remember the function name. Don't guess: `find` supports Chinese, English, abbreviations, and even backend tool names. Better yet, each result comes with the method's **contract**: `requires` (what inputs are needed) and `produces` (what it outputs). One glance at the contract tells you that `did` is not ready-to-use—it requires you to first declare a design and pass a parallel trends test.

In [3]:
for r in sv.registry.find("双重差分"):
    print(r["full_name"].split(".")[-1])
    print("   需要 (requires):", r["requires"])
    print("   产出 (produces):", r["produces"])
    print("   后端:", r["key_tools"])

did
   需要 (requires): {'design': ['panel_id', 'time', 'treatment'], 'variables': ['outcome'], 'identification': ['parallel_trends']}
   产出 (produces): {'models': ['did', 'twfe'], 'diagnostics': ['robustness']}
   后端: ['statsmodels', 'linearmodels', 'numpy']


## Laying Out the Entire Workflow: `resolve_plan`

The prerequisites for `did`—declaring a design, passing a parallel trends test—themselves have prerequisites. Rather than mentally mapping that dependency graph, let the registry do it: `resolve_plan("did")` recursively takes all steps needed to run `did` and topologically sorts them into an executable chain in the correct order.

It also surfaces two categories of information you should watch for. `needs_input` is whatever the registry cannot automatically produce from other functions—things you (the researcher) must provide, such as "what target estimand do you want" (`estimand.target`) or "which column is the outcome variable" (`variables.outcome`). `escalations` are steps involving causal assumptions that should not be silently filled in by the tool but require human confirmation instead. This is the right balance for social-science analysis: automate what can be automated, leave judgment calls to people.

In [4]:
plan = sv.registry.resolve_plan("did")
print("执行顺序:", [p.split(".")[-1] for p in plan["plan"]])
print("需要你提供的输入 (needs_input):")
for ni in plan["needs_input"]:
    print("   -", ni["slot"] + "." + ni["key"])
print(f"需人工确认的步骤 (escalations): {len(plan['escalations'])} 处")

执行顺序: ['ingest', 'declare_design', 'parallel_trends', 'did']
需要你提供的输入 (needs_input):
   - variables.outcome
   - estimand.target
   - variables.outcome
需人工确认的步骤 (escalations): 9 处


Visualizing this chain is clearer. The diagram below is not decorative: each box is a step from `resolve_plan`, the label below shows which slot in the state it populates, and the bottom notes the one manual input you must provide. The entire sequence is derived automatically from the registry based on each method's `requires ↔ produces` relationships.

In [5]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

# 让图里的中文正常显示:挑一个系统里存在的 CJK 字体(找不到就退回默认)
_have = {f.name for f in font_manager.fontManager.ttflist}
for _cjk in ("Heiti TC", "Songti SC", "Arial Unicode MS", "Hiragino Sans GB",
             "PingFang SC", "Noto Sans CJK SC", "STHeiti"):
    if _cjk in _have:
        plt.rcParams["font.sans-serif"] = [_cjk]
        plt.rcParams["axes.unicode_minus"] = False
        break

steps = [p.split(".")[-1] for p in plan["plan"]]
produced = {   # 每一步补上的关键状态格(教学标注)
    "ingest": "载入数据",
    "declare_design": "声明设计\npanel_id/time/treatment",
    "parallel_trends": "识别假设\n平行趋势",
    "did": "估计 ATT\nmodels + diagnostics",
}

fig, ax = plt.subplots(figsize=(11, 3.0))
ax.set_xlim(0, len(steps) * 3.2)
ax.set_ylim(0, 3)
ax.axis("off")

for i, s in enumerate(steps):
    x = i * 3.2 + 0.2
    box = FancyBboxPatch((x, 1.05), 2.4, 0.95, boxstyle="round,pad=0.08",
                         linewidth=1.6, edgecolor="#2b6cb0", facecolor="#ebf4ff")
    ax.add_patch(box)
    ax.text(x + 1.2, 1.72, f"{i+1}. {s}", ha="center", va="center",
            fontsize=11, weight="bold", color="#1a365d")
    ax.text(x + 1.2, 1.28, produced[s], ha="center", va="center",
            fontsize=7.5, color="#2c5282")
    if i < len(steps) - 1:
        arr = FancyArrowPatch((x + 2.4, 1.55), (x + 3.2 + 0.2, 1.55),
                              arrowstyle="-|>", mutation_scale=16, color="#4a5568")
        ax.add_patch(arr)

ax.text(0.2, 2.6, "resolve_plan('did') 自动排出的执行链",
        fontsize=12, weight="bold", color="#1a202c")
ax.text(0.2, 0.5, "唯一需你给定的输入:  estimand['target'] = 'ATT'   ·   variables['outcome'] = 'y'",
        fontsize=9, color="#c05621", style="italic")

plt.tight_layout()
fig.savefig("fig_resolve_plan_did.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("saved -> fig_resolve_plan_did.png")

saved -> fig_resolve_plan_did.png


![DID Execution Chain](fig_resolve_plan_did.png)

## Learning StudyState: Everything from One Analysis Goes Here

All the `requires` / `produces` relationships above use the same set of "slot names": `design`, `variables`, `identification`, `models`, and so on. These slots come from `StudyState`—the library's unified analysis object. A complete research project, from raw data to final deliverables, is organized into its slots. In bioinformatics, AnnData holds a fixed set of fields for an expression matrix; here it's different. Social-science data are inherently incommensurable—a survey, a text corpus, a social network cannot fit into the same matrix—so `StudyState` unifies not data itself but **vocabulary**: it gives "design," "assumption," and "model" common names so that machine can check dependencies.

Below are the 12 slots and what each holds. This tutorial's DID analysis uses only some of them (`sources` / `design` / `variables` / `identification` / `models` / `diagnostics`); later tutorials will populate slots for qualitative coding, documentary evidence, governance compliance, and more.

In [6]:
for name, (meaning, keys) in sv.SLOTS.items():
    print(f"  {name:15s} {meaning}")

  sources         已登记的原始输入:数据集/语料/手稿/.bib/扫描件
  design          研究设计:抽样框/权重/分层/聚类/panel_id/time/处理时点/分析单元
  variables       codebook:变量定义/类型/测量层次/量表题项
  corpus          文本即数据态:文档/分词/dfm/OCR文本/TEI/编码单元
  codes           质性编码态:质性codebook/编码片段/主题/主题地图
  estimand        目标量:ATT/患病率/关联/目标总体(通常由用户/研究问题给定)
  identification  识别假设:DAG/平行趋势/IV有效性/排他性/正值性
  models          拟合结果:回归表/DID/FE/event-study/加权估计/主题模型/网络
  diagnostics     检验:pretrend/平衡性/稳健性矩阵/信度α/拟合/敏感性
  evidence        证据链:claim→引语/引文/quote-trace索引/已核验.bib/来源span
  governance      伦理合规:IRB/知情同意/PII去标识/数据使用许可/AI使用披露
  artifacts       交付物:图/docx-pdf稿件/表/TEI-XML/apparatus


## Loading Data

Now for the actual analysis. Use the built-in loader to fetch the synthetic panel and see what it looks like. The data is in long format (one row per "firm × year"): `firm_id` is the unit, `year` is the time index, `treat_post` flags whether the observation has been treated, `first_treated` is the year each firm was first exposed, and `y` is the outcome variable.

In [7]:
df = ds.load_did_panel(att=-0.8)   # 真实 ATT 设为 -0.8,便于对照
print("面板维度:", df.shape)
df.head()

面板维度: (320, 8)


,firm_id,year,treat,post,treat_post,first_treated,y,x1
0,0,2010,1,0,0,2015,1.6221,1.5139
1,0,2011,1,0,0,2015,2.8355,0.7813
2,0,2012,1,0,0,2015,2.5744,-0.3139
3,0,2013,1,0,0,2015,3.2232,1.9603
4,0,2014,1,0,0,2015,3.5321,1.3151


## Ingesting Data and Declaring Design

The first step of analysis is not fitting a model—it's telling the tool "what role does each column play." `ingest` registers the data in the `sources` slot of the state; `declare_design` writes the panel ID, time index, treatment indicator, and treatment onset year into the `design` slot. Declare once, and every causal function reads from here—no need to pass parameters repeatedly. Don't forget to also populate `estimand` (we want the average treatment effect on the treated, ATT, not mere association) and `variables.outcome` (the outcome is `y`)—these are exactly the inputs that `resolve_plan` told you to provide.

In [8]:
st = sv.StudyState()
st.write("estimand", "target", "ATT")   # 你想估的量:平均处理效应
st.write("variables", "outcome", "y")   # 结果变量

sv.pp.ingest(st, data=df)
sv.pp.declare_design(
    st,
    panel_id="firm_id",
    time="year",
    treatment="treat_post",
    first_treated="first_treated",
)
st.design

Slot(panel_id, time, treatment, first_treated)

## Testing the Prerequisite: Parallel Trends

Whether DID can be read as causal hinges on one key assumption: **parallel trends**—absent the policy, treated and control units would have evolved on parallel trajectories. This cannot be tested directly, but can be indirectly examined using "pre-trends" in periods before treatment. `parallel_trends` fits an event study and jointly tests all pre-treatment relative-period coefficients: the null is "all pre-treatment coefficients equal zero." If `p > 0.05`, we fail to reject parallel trends and the assumption holds; if `p` is small, pre-trends are already diverging, and any coefficient estimated downstream should not be called causal.

In [9]:
sv.tl.parallel_trends(st)

pt = st.diagnostics["pretrend"]
print("平行趋势判定:", st.identification["parallel_trends"])
print(f"联合 F = {pt['joint_F']:.2f}   p = {pt['p_value']:.3f}   (前导期数 = {pt['n_pre']})")

平行趋势判定: pass
联合 F = 0.47   p = 0.755   (前导期数 = 4)


The `p` value is well above 0.05—the pre-treatment coefficients are jointly insignificant and parallel trends hold. Estimation can proceed. This verdict is recorded in the state's `identification` slot, serving as the prerequisite for the next step, `did`.

## Estimating ATT

Now that the prerequisite is met, estimation can proceed. `did` fits `y ~ treat_post + unit fixed effects + time fixed effects` and computes robust standard errors clustered at the unit level (inference on treatment effects typically requires clustering at the unit level). It also reads the parallel trends verdict from the previous step into the conclusion: if it passes, the result is labeled "causal ATT"; if not, it is automatically downgraded to "association, not causal"—this step does not beautify results for you.

In [10]:
sv.tl.did(st)

m = st.models["did"]
print(f"ATT   = {m['att']:.3f}")
print(f"95%CI = [{m['ci'][0]:.3f}, {m['ci'][1]:.3f}]")
print(f"SE    = {m['se']:.3f}   (聚类于 {m['n_clusters']} 家企业)")
print(f"p     = {m['p']:.2e}")
print("平行趋势:", m["parallel_trends"], " · 估计量:", m["estimator"])

ATT   = -0.731
95%CI = [-0.931, -0.531]
SE    = 0.102   (聚类于 40 家企业)
p     = 8.15e-13
平行趋势: pass  · 估计量: twfe_ols_cluster


The estimated ATT ≈ −0.73, with a 95% confidence interval [−0.93, −0.53] that excludes zero and contains the true value −0.8. The policy caused a significant decrease in the outcome. At this point, a minimal but complete causal analysis chain has been executed.

## What the State Contains Now

After the analysis completes, look back at what `StudyState` now holds. `populated()` lists all slots and keys that have been filled—from raw data (`sources`), to design (`design`), target estimand (`estimand`), outcome variables (`variables`), identification assumptions (`identification`), through models (`models`) and diagnostics (`diagnostics`). This is the value of a unified analysis object: the entire scope of one study is in one place, and anyone who wants to extend it or audit it can open it and see everything at a glance.

In [11]:
for slot, keys in st.populated().items():
    print(f"  {slot:15s} {keys}")

  sources         ['datasets', 'dataset_name', 'dataset_meta']
  design          ['panel_id', 'time', 'treatment', 'first_treated']
  variables       ['outcome']
  estimand        ['target']
  identification  ['parallel_trends']
  models          ['twfe', 'did']
  diagnostics     ['pretrend', 'robustness']


## Two Mechanisms That Make Analysis More Robust

Above we naturally used the registry; now let's highlight two concrete benefits it brings—they are precisely what organizing an analysis as "state + registry" adds beyond a plain script.

First, **prerequisite checks will actually stop you**. If you call `did` on an unprepared, empty state, it will not silently hand you a plausible-looking wrong result; instead it raises an error and tells you exactly which slot is missing and which function should fill it. This prevents "computing a number on the wrong state that sounds just believable enough to fool yourself."

In [12]:
empty = sv.StudyState()
try:
    sv.tl.did(empty)   # 空状态:必然被拒
except sv.RegistryError as e:
    print(e)

socialverse.tl._causal.did cannot run — unmet requires:
  - design.panel_id (produced by: declare_design)
  - design.time (produced by: declare_design)
  - design.treatment (produced by: declare_design)
  - variables.outcome (user-supplied input)
  - identification.parallel_trends (produced by: parallel_trends)
Query registry.get_prerequisites(...) or registry.resolve_plan(...) to plan the chain.


Second, **every step is logged in a reproducible audit trail**. As the analysis runs, the state automatically accumulates a `provenance` ledger: which step, which function was used, what was consumed, what was produced. In social science, "where the conclusion came from—which step, which dataset" is often as important as the conclusion itself—this ledger gives an analysis a built-in audit trail, so others can reproduce it without asking you again. `st.summary()` displays the state's full contents plus the ledger of steps all at once.

In [13]:
for rec in st.provenance:
    req = ", ".join(f"{s}{ks}" for s, ks in rec["requires"].items()) or "(无)"
    pro = ", ".join(f"{s}{ks}" for s, ks in rec["produces"].items()) or "(无)"
    print(f"  第 {rec['step']} 步: {rec['function'].split('.')[-1]}")
    print(f"          消费: {req}")
    print(f"          产出: {pro}")

print()
print(st.summary())

  第 1 步: ingest
          消费: (无)
          产出: sources['datasets']
  第 2 步: declare_design
          消费: sources['datasets']
          产出: design['panel_id', 'time', 'treatment', 'first_treated', 'weights', 'strata', 'psu', 'unit']
  第 3 步: parallel_trends
          消费: design['panel_id', 'time', 'treatment', 'first_treated'], variables['outcome'], estimand['target']
          产出: diagnostics['pretrend'], identification['parallel_trends']
  第 4 步: did
          消费: design['panel_id', 'time', 'treatment'], variables['outcome'], identification['parallel_trends']
          产出: models['did', 'twfe'], diagnostics['robustness']

StudyState
  sources: datasets, dataset_name, dataset_meta
  design: panel_id, time, treatment, first_treated
  variables: outcome
  estimand: target
  identification: parallel_trends
  models: twfe, did
  diagnostics: pretrend, robustness
  provenance: 4 step(s)


## Summary

We've used one minimal DID analysis to tour the two pillars of `socialverse`: `StudyState` is the social-science equivalent of a unified analysis object, where a study's data, design, assumptions, models, and diagnostics are organized into slots; the registry is like a methods handbook that sequences your workflow (`resolve_plan`), helps you find methods (`find`), and catches errors (prerequisite checks). The whole "registry + query" idea parallels `ov.registry` in bioinformatics, but `StudyState` differs from AnnData in one key way—it unifies **vocabulary** rather than data, because social-science surveys, corpora, and networks simply cannot fit into the same matrix. Compared to a plain estimation script, what you gain is a prerequisite gate that actually stops you, and an audit trail of reproducible evidence that runs throughout.

The next tutorial, [02_causal_did](02_causal_did.ipynb), will expand on this causal chain: the joint test of parallel trends, event studies for dynamic effects, robustness of standard errors, and publication-quality figures.